# PySpark practice 8: window functions

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-practice-08-window-functions.ipynb)

16 window-function exercises: top-N, ties, deduplication, lag, running and moving metrics, fills, sessions, streaks and SQL. Each exercise names the [reference entries](pyspark-reference-08-window-functions.ipynb) it practises, shows its inputs and the exact expected output, and hides a reference solution you can reveal when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items`, `products` and `events` tables (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry), and the helpers from [PySpark reference 8](pyspark-reference-08-window-functions.ipynb#setup-and-sample-data). Exercises that need other data create it in their input cell.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)), (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)), (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)), (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)), (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)), (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)), (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

import contextlib
import io
import logging
import re


def plan_lines(df, mode="simple"):
    """The lines df.explain(mode) prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue().splitlines()


def final_plan(df):
    """Run df, then return the plan Spark actually used (AQE's final plan when there is one)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    return text


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        print(type(error).__name__ + ":", str(error).split(".")[0][:100])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

## Exercises

5 easy, 8 medium, 3 hard. Write your answer in the empty cell under each exercise as a DataFrame called `result`, run it, and compare it with the expected output (columns, values and row order).

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Top two orders per customer](#exercise-1-top-two-orders-per-customer) | Easy | [Keep the top N rows per group](pyspark-reference-08-window-functions.ipynb#keep-the-top-n-rows-per-group) |
| 2 | [Rank with ties three ways](#exercise-2-rank-with-ties-three-ways) | Easy | [`rank`, `dense_rank`](pyspark-reference-08-window-functions.ipynb#rank-dense_rank), [`row_number`](pyspark-reference-08-window-functions.ipynb#row_number) |
| 3 | [Second highest amount per customer](#exercise-3-second-highest-amount-per-customer) | Medium | [`rank`, `dense_rank`](pyspark-reference-08-window-functions.ipynb#rank-dense_rank) |
| 4 | [Latest version per key from a CDC feed](#exercise-4-latest-version-per-key-from-a-cdc-feed) | Medium | [Deduplicate: keep the latest record per key](pyspark-reference-08-window-functions.ipynb#deduplicate-keep-the-latest-record-per-key) |
| 5 | [Change from the previous order](#exercise-5-change-from-the-previous-order) | Easy | [`lag`, `lead`](pyspark-reference-08-window-functions.ipynb#lag-lead) |
| 6 | [Running total and share of customer revenue](#exercise-6-running-total-and-share-of-customer-revenue) | Easy | [Running totals and cumulative metrics](pyspark-reference-08-window-functions.ipynb#running-totals-and-cumulative-metrics) |
| 7 | [Seven-day moving average with missing days](#exercise-7-seven-day-moving-average-with-missing-days) | Medium | [Moving windows: last N rows vs last N days](pyspark-reference-08-window-functions.ipynb#moving-windows-last-n-rows-vs-last-n-days) |
| 8 | [Forward fill sparse readings](#exercise-8-forward-fill-sparse-readings) | Medium | [Fill gaps: forward fill and back fill](pyspark-reference-08-window-functions.ipynb#fill-gaps-forward-fill-and-back-fill) |
| 9 | [Sessionize clicks with a 30-minute gap](#exercise-9-sessionize-clicks-with-a-30-minute-gap) | Medium | [Sessionize events](pyspark-reference-08-window-functions.ipynb#sessionize-events) |
| 10 | [Longest login streak](#exercise-10-longest-login-streak) | Hard | [Find consecutive runs: gaps and islands](pyspark-reference-08-window-functions.ipynb#find-consecutive-runs-gaps-and-islands) |
| 11 | [Number runs of unchanged status](#exercise-11-number-runs-of-unchanged-status) | Hard | [Detect changes and number the runs](pyspark-reference-08-window-functions.ipynb#detect-changes-and-number-the-runs) |
| 12 | [Each customer's second order](#exercise-12-each-customers-second-order) | Medium | [`nth_value`](pyspark-reference-08-window-functions.ipynb#nth_value) |
| 13 | [Quartiles and percentile rank](#exercise-13-quartiles-and-percentile-rank) | Medium | [`ntile`](pyspark-reference-08-window-functions.ipynb#ntile), [`percent_rank`, `cume_dist`](pyspark-reference-08-window-functions.ipynb#percent_rank-cume_dist) |
| 14 | [Running count of distinct products](#exercise-14-running-count-of-distinct-products) | Hard | [Aggregates over a window](pyspark-reference-08-window-functions.ipynb#aggregates-over-a-window) |
| 15 | [Orders above their customer's average](#exercise-15-orders-above-their-customers-average) | Easy | [Aggregates over a whole partition](pyspark-reference-08-window-functions.ipynb#aggregates-over-a-whole-partition) |
| 16 | [Top order per customer in SQL](#exercise-16-top-order-per-customer-in-sql) | Medium | [Window functions in Spark SQL](pyspark-reference-08-window-functions.ipynb#window-functions-in-spark-sql) |

## Exercise 1: Top two orders per customer

**Difficulty:** Easy · about 5 min · **Practises:** [Keep the top N rows per group](pyspark-reference-08-window-functions.ipynb#keep-the-top-n-rows-per-group)

*Top-N per group is the most-asked window question in interviews.*

Return `customer_id`, `order_id` and `amount` of each customer's two largest orders (ties broken by `order_id`), ordered by `customer_id`, `amount` descending.

In [ ]:
orders.select("order_id", "customer_id", "amount").show()

**Expected output**

| customer_id | order_id | amount |
| --- | --- | --- |
| 1 | 1 | 120.00 |
| 1 | 2 | 84.50 |
| 2 | 4 | 260.00 |
| 2 | 5 | 55.25 |
| 3 | 7 | 310.00 |
| 3 | 6 | 149.00 |
| 4 | 9 | 1180.00 |
| 4 | 8 | 89.90 |
| 5 | 10 | 74.00 |
| 6 | 11 | 210.00 |
| 6 | 12 | 96.40 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
spec = Window.partitionBy("customer_id").orderBy(F.desc("amount"), "order_id")
result = (orders.withColumn("rn", F.row_number().over(spec)).where("rn <= 2")
          .select("customer_id", "order_id", "amount").orderBy("customer_id", F.desc("amount")))
result.show()

## Exercise 2: Rank with ties three ways

**Difficulty:** Easy · about 5 min · **Practises:** [`rank`, `dense_rank`](pyspark-reference-08-window-functions.ipynb#rank-dense_rank), [`row_number`](pyspark-reference-08-window-functions.ipynb#row_number)

*Knowing exactly how row_number, rank and dense_rank differ on ties.*

Rank `scores` by `points` descending within each `team`. Return `team`, `player`, `points`, `row_number` (ties broken by player), `rank` and `dense_rank`, ordered by `team`, `row_number`.

In [ ]:
scores = spark.createDataFrame([("A", "ana", 30), ("A", "ben", 30), ("A", "cai", 20), ("A", "dee", 10), ("B", "eli", 25), ("B", "fay", 25)], "team string, player string, points int")
scores.show()

**Expected output**

| team | player | points | row_number | rank | dense_rank |
| --- | --- | --- | --- | --- | --- |
| A | ana | 30 | 1 | 1 | 1 |
| A | ben | 30 | 2 | 1 | 1 |
| A | cai | 20 | 3 | 3 | 2 |
| A | dee | 10 | 4 | 4 | 3 |
| B | eli | 25 | 1 | 1 | 1 |
| B | fay | 25 | 2 | 1 | 1 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
spec = Window.partitionBy("team").orderBy(F.desc("points"))
result = scores.select(
    "team", "player", "points",
    F.row_number().over(spec.orderBy(F.desc("points"), "player")).alias("row_number"),
    F.rank().over(spec).alias("rank"),
    F.dense_rank().over(spec).alias("dense_rank"),
).orderBy("team", "row_number")
result.show()

## Exercise 3: Second highest amount per customer

**Difficulty:** Medium · about 10 min · **Practises:** [`rank`, `dense_rank`](pyspark-reference-08-window-functions.ipynb#rank-dense_rank)

*"Nth highest salary" in its data engineering form; ties must not hide the second value.*

Return `customer_id` and `second_highest` (the second-highest **distinct** amount among the customer's orders) for customers who have one, ordered by `customer_id`.

In [ ]:
extra = orders.unionByName(spark.createDataFrame([(13, 3, "paid", datetime(2026, 1, 15), D("310.00"))], orders.schema))
extra.select("order_id", "customer_id", "amount").orderBy("customer_id").show()

**Expected output**

| customer_id | second_highest |
| --- | --- |
| 1 | 84.50 |
| 2 | 55.25 |
| 3 | 149.00 |
| 4 | 89.90 |
| 6 | 96.40 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
spec = Window.partitionBy("customer_id").orderBy(F.desc("amount"))
result = (extra.withColumn("dr", F.dense_rank().over(spec)).where("dr = 2")
          .select("customer_id", F.col("amount").alias("second_highest")).distinct().orderBy("customer_id"))
result.show()

## Exercise 4: Latest version per key from a CDC feed

**Difficulty:** Medium · about 10 min · **Practises:** [Deduplicate: keep the latest record per key](pyspark-reference-08-window-functions.ipynb#deduplicate-keep-the-latest-record-per-key)

*Silver tables keep one current row per key, chosen reproducibly.*

Keep the latest change per `order_id` by `lsn`, breaking ties by the later `ingested_at`. Return `order_id`, `status`, `lsn`, ordered by `order_id`.

In [ ]:
cdc = spark.createDataFrame([(101, "placed", 1, datetime(2026, 1, 5, 9, 0)), (101, "paid", 2, datetime(2026, 1, 5, 9, 5)), (102, "placed", 1, datetime(2026, 1, 5, 10, 0)),
                             (102, "cancelled", 2, datetime(2026, 1, 5, 10, 1)), (102, "refunded", 2, datetime(2026, 1, 5, 10, 9))], "order_id int, status string, lsn int, ingested_at timestamp")
cdc.show()

**Expected output**

| order_id | status | lsn |
| --- | --- | --- |
| 101 | paid | 2 |
| 102 | refunded | 2 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
spec = Window.partitionBy("order_id").orderBy(F.desc("lsn"), F.desc("ingested_at"))
result = cdc.withColumn("rn", F.row_number().over(spec)).where("rn = 1").select("order_id", "status", "lsn").orderBy("order_id")
result.show()

## Exercise 5: Change from the previous order

**Difficulty:** Easy · about 5 min · **Practises:** [`lag`, `lead`](pyspark-reference-08-window-functions.ipynb#lag-lead)

*Month-over-month and order-over-order deltas are lag calculations.*

For customers 1 to 3, return `order_id`, `amount`, `prev_amount` and `pct_change` (relative to the previous order, ×100, rounded to 1), ordered by `order_id`.

In [ ]:
orders.where("customer_id <= 3").select("order_id", "customer_id", "order_ts", "amount").show()

**Expected output**

| order_id | amount | prev_amount | pct_change |
| --- | --- | --- | --- |
| 1 | 120.00 | null | null |
| 2 | 84.50 | 120.00 | -29.6 |
| 3 | 30.00 | 84.50 | -64.5 |
| 4 | 260.00 | null | null |
| 5 | 55.25 | 260.00 | -78.8 |
| 6 | 149.00 | null | null |
| 7 | 310.00 | 149.00 | 108.1 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
prev = F.lag("amount").over(Window.partitionBy("customer_id").orderBy("order_ts"))
result = (orders.where("customer_id <= 3")
          .select("order_id", "amount", prev.alias("prev_amount"), F.round((F.col("amount") - prev) / prev * 100, 1).alias("pct_change"))
          .orderBy("order_id"))
result.show()

## Exercise 6: Running total and share of customer revenue

**Difficulty:** Easy · about 5 min · **Practises:** [Running totals and cumulative metrics](pyspark-reference-08-window-functions.ipynb#running-totals-and-cumulative-metrics)

*Cumulative revenue curves per customer.*

For customers 1 and 4, return `order_id`, `running_total` and `cum_pct` (running total as a percentage of the customer's total, rounded to 1), ordered by `order_id`.

In [ ]:
orders.where("customer_id IN (1, 4)").select("order_id", "customer_id", "order_ts", "amount").show()

**Expected output**

| order_id | running_total | cum_pct |
| --- | --- | --- |
| 1 | 120.00 | 51.2 |
| 2 | 204.50 | 87.2 |
| 3 | 234.50 | 100.0 |
| 8 | 89.90 | 7.1 |
| 9 | 1269.90 | 100.0 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
ordered = Window.partitionBy("customer_id").orderBy("order_ts", "order_id")
running = F.sum("amount").over(ordered.rowsBetween(Window.unboundedPreceding, 0))
result = (orders.where("customer_id IN (1, 4)")
          .select("order_id", running.alias("running_total"), F.round(running / F.sum("amount").over(Window.partitionBy("customer_id")) * 100, 1).alias("cum_pct"))
          .orderBy("order_id"))
result.show()

## Exercise 7: Seven-day moving average with missing days

**Difficulty:** Medium · about 15 min · **Practises:** [Moving windows: last N rows vs last N days](pyspark-reference-08-window-functions.ipynb#moving-windows-last-n-rows-vs-last-n-days)

*Dashboards smooth daily metrics; gaps must not stretch the window.*

From daily revenue, return `day`, `revenue`, `avg_3_rows` and `avg_3_days` (the current day and the two calendar days before it), both rounded to 2, ordered by `day`.

In [ ]:
daily = spark.createDataFrame([(date(2026, 1, 1), 100), (date(2026, 1, 2), 200), (date(2026, 1, 3), 300), (date(2026, 1, 5), 500), (date(2026, 1, 6), 600)], "day date, revenue int")
daily.show()

**Expected output**

| day | revenue | avg_3_rows | avg_3_days |
| --- | --- | --- | --- |
| 2026-01-01 | 100 | 100.0 | 100.0 |
| 2026-01-02 | 200 | 150.0 | 150.0 |
| 2026-01-03 | 300 | 200.0 | 200.0 |
| 2026-01-05 | 500 | 333.33 | 400.0 |
| 2026-01-06 | 600 | 466.67 | 550.0 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
rows = Window.orderBy("day").rowsBetween(-2, 0)
days = Window.orderBy(F.unix_date("day")).rangeBetween(-2, 0)
result = daily.select("day", "revenue", F.round(F.avg("revenue").over(rows), 2).alias("avg_3_rows"),
                      F.round(F.avg("revenue").over(days), 2).alias("avg_3_days")).orderBy("day")
result.show()

## Exercise 8: Forward fill sparse readings

**Difficulty:** Medium · about 10 min · **Practises:** [Fill gaps: forward fill and back fill](pyspark-reference-08-window-functions.ipynb#fill-gaps-forward-fill-and-back-fill)

*IoT and price feeds send values only when they change.*

Forward fill `temp` per `sensor` by `minute`; where no earlier value exists, back fill instead. Return `sensor`, `minute`, `filled`, ordered by `sensor`, `minute`.

In [ ]:
readings = spark.createDataFrame([("S1", 1, None), ("S1", 2, 20.5), ("S1", 3, None), ("S1", 4, 22.0), ("S2", 1, None), ("S2", 2, None), ("S2", 3, 18.0)], "sensor string, minute int, temp double")
readings.show()

**Expected output**

| sensor | minute | filled |
| --- | --- | --- |
| S1 | 1 | 20.5 |
| S1 | 2 | 20.5 |
| S1 | 3 | 20.5 |
| S1 | 4 | 22.0 |
| S2 | 1 | 18.0 |
| S2 | 2 | 18.0 |
| S2 | 3 | 18.0 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
w = Window.partitionBy("sensor").orderBy("minute")
forward = F.last("temp", ignorenulls=True).over(w.rowsBetween(Window.unboundedPreceding, 0))
backward = F.first("temp", ignorenulls=True).over(w.rowsBetween(0, Window.unboundedFollowing))
result = readings.select("sensor", "minute", F.coalesce(forward, backward).alias("filled")).orderBy("sensor", "minute")
result.show()

## Exercise 9: Sessionize clicks with a 30-minute gap

**Difficulty:** Medium · about 15 min · **Practises:** [Sessionize events](pyspark-reference-08-window-functions.ipynb#sessionize-events)

*The lag + flag + running-sum recipe is asked in most analytics interviews.*

Assign each click a `session` number per user (a new session starts after more than 30 minutes of inactivity). Return `user_id`, `ts`, `session`, ordered by `user_id`, `ts`.

In [ ]:
clicks = spark.createDataFrame([("u1", datetime(2026, 1, 6, 9, 0)), ("u1", datetime(2026, 1, 6, 9, 20)), ("u1", datetime(2026, 1, 6, 10, 0)),
                                ("u2", datetime(2026, 1, 6, 9, 0)), ("u2", datetime(2026, 1, 6, 9, 31))], "user_id string, ts timestamp")
clicks.show()

**Expected output**

| user_id | ts | session |
| --- | --- | --- |
| u1 | 2026-01-06 09:00:00 | 1 |
| u1 | 2026-01-06 09:20:00 | 1 |
| u1 | 2026-01-06 10:00:00 | 2 |
| u2 | 2026-01-06 09:00:00 | 1 |
| u2 | 2026-01-06 09:31:00 | 2 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
w = Window.partitionBy("user_id").orderBy("ts")
gap = (F.col("ts").cast("long") - F.lag("ts").over(w).cast("long")) / 60
new_session = F.when(gap.isNull() | (gap > 30), 1).otherwise(0)
result = (clicks.withColumn("new", new_session)
          .withColumn("session", F.sum("new").over(w.rowsBetween(Window.unboundedPreceding, 0)))
          .select("user_id", "ts", "session").orderBy("user_id", "ts"))
result.show()

## Exercise 10: Longest login streak

**Difficulty:** Hard · about 15 min · **Practises:** [Find consecutive runs: gaps and islands](pyspark-reference-08-window-functions.ipynb#find-consecutive-runs-gaps-and-islands)

*Gaps-and-islands is a classic hard interview question.*

Return each user's longest streak of consecutive login days as `user_id`, `days`, `start`, `end` (earliest streak on ties), ordered by `user_id`.

In [ ]:
logins = spark.createDataFrame([("u1", "2026-01-01"), ("u1", "2026-01-02"), ("u1", "2026-01-02"), ("u1", "2026-01-04"), ("u1", "2026-01-05"), ("u1", "2026-01-06"),
                                ("u2", "2026-01-10"), ("u3", "2026-01-01"), ("u3", "2026-01-03")], "user_id string, login_date string")
logins.show()

**Expected output**

| user_id | days | start | end |
| --- | --- | --- | --- |
| u1 | 3 | 2026-01-04 | 2026-01-06 |
| u2 | 1 | 2026-01-10 | 2026-01-10 |
| u3 | 1 | 2026-01-01 | 2026-01-01 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
days = logins.select("user_id", F.to_date("login_date").alias("day")).distinct()
islands = days.withColumn("grp", F.date_sub("day", F.row_number().over(Window.partitionBy("user_id").orderBy("day"))))
streaks = islands.groupBy("user_id", "grp").agg(F.count("*").alias("days"), F.min("day").alias("start"), F.max("day").alias("end"))
best = Window.partitionBy("user_id").orderBy(F.desc("days"), "start")
result = streaks.withColumn("rn", F.row_number().over(best)).where("rn = 1").select("user_id", "days", "start", "end").orderBy("user_id")
result.show()

## Exercise 11: Number runs of unchanged status

**Difficulty:** Hard · about 15 min · **Practises:** [Detect changes and number the runs](pyspark-reference-08-window-functions.ipynb#detect-changes-and-number-the-runs)

*Uptime reports turn row-level states into intervals.*

For each machine, return `machine`, `state`, `from_minute` and `to_minute` for every run of consecutive minutes with the same state, ordered by `machine`, `from_minute`.

In [ ]:
status_log = spark.createDataFrame([("M1", 1, "up"), ("M1", 2, "up"), ("M1", 3, "down"), ("M1", 4, "up"), ("M1", 5, "up"), ("M2", 1, "down"), ("M2", 2, "down")],
                                   "machine string, minute int, state string")
status_log.show()

**Expected output**

| machine | state | from_minute | to_minute |
| --- | --- | --- | --- |
| M1 | up | 1 | 2 |
| M1 | down | 3 | 3 |
| M1 | up | 4 | 5 |
| M2 | down | 1 | 2 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
w = Window.partitionBy("machine").orderBy("minute")
changed = F.when(F.lag("state").over(w).eqNullSafe(F.col("state")), 0).otherwise(1)
runs = status_log.withColumn("run", F.sum(changed).over(w.rowsBetween(Window.unboundedPreceding, 0)))
result = runs.groupBy("machine", "run", "state").agg(F.min("minute").alias("from_minute"), F.max("minute").alias("to_minute")).select("machine", "state", "from_minute", "to_minute").orderBy("machine", "from_minute")
result.show()

## Exercise 12: Each customer's second order

**Difficulty:** Medium · about 10 min · **Practises:** [`nth_value`](pyspark-reference-08-window-functions.ipynb#nth_value)

*Second-purchase rate is a key retention metric.*

Return every customer with orders as `customer_id`, `first_order_id` and `second_order_id` (null when there is only one order), ordered by `customer_id`.

In [ ]:
orders.select("order_id", "customer_id", "order_ts").show()

**Expected output**

| customer_id | first_order_id | second_order_id |
| --- | --- | --- |
| 1 | 1 | 2 |
| 2 | 4 | 5 |
| 3 | 6 | 7 |
| 4 | 8 | 9 |
| 5 | 10 | null |
| 6 | 11 | 12 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
w = Window.partitionBy("customer_id").orderBy("order_ts", "order_id").rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
result = (orders.select("customer_id", F.first("order_id").over(w).alias("first_order_id"), F.nth_value("order_id", 2).over(w).alias("second_order_id"))
          .distinct().orderBy("customer_id"))
result.show()

## Exercise 13: Quartiles and percentile rank

**Difficulty:** Medium · about 10 min · **Practises:** [`ntile`](pyspark-reference-08-window-functions.ipynb#ntile), [`percent_rank`, `cume_dist`](pyspark-reference-08-window-functions.ipynb#percent_rank-cume_dist)

*Customer segmentation by spend.*

Rank customers by total revenue: return `customer_id`, `revenue`, `quartile` (`ntile(4)`, highest revenue in quartile 1) and `pct_rank` (rounded to 2, highest revenue = 0), ordered by `revenue` descending.

In [ ]:
orders.groupBy("customer_id").agg(F.sum("amount").alias("revenue")).show()

**Expected output**

| customer_id | revenue | quartile | pct_rank |
| --- | --- | --- | --- |
| 4 | 1269.90 | 1 | 0.0 |
| 3 | 459.00 | 1 | 0.2 |
| 2 | 315.25 | 2 | 0.4 |
| 6 | 306.40 | 2 | 0.6 |
| 1 | 234.50 | 3 | 0.8 |
| 5 | 74.00 | 4 | 1.0 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
revenue = orders.groupBy("customer_id").agg(F.sum("amount").alias("revenue"))
w = Window.orderBy(F.desc("revenue"))
result = revenue.select("customer_id", "revenue", F.ntile(4).over(w).alias("quartile"), F.round(F.percent_rank().over(w), 2).alias("pct_rank")).orderBy(F.desc("revenue"))
result.show()

## Exercise 14: Running count of distinct products

**Difficulty:** Hard · about 15 min · **Practises:** [Aggregates over a window](pyspark-reference-08-window-functions.ipynb#aggregates-over-a-window)

*`count(DISTINCT)` is not allowed over a window; interviewers check whether you know the workaround.*

For each order line of customers 1 and 2 in time order, return `order_id`, `sku` and `distinct_skus_so_far` (distinct SKUs the customer has bought up to and including that line). Order by `order_id`, `sku`.

In [ ]:
order_items.join(orders.select("order_id", "customer_id", "order_ts"), "order_id").where("customer_id <= 2").show()

**Expected output**

| order_id | sku | distinct_skus_so_far |
| --- | --- | --- |
| 1 | SKU-4001 | 1 |
| 1 | SKU-4007 | 2 |
| 2 | SKU-4002 | 3 |
| 2 | SKU-4009 | 4 |
| 3 | SKU-4002 | 4 |
| 4 | SKU-4003 | 1 |
| 4 | SKU-4004 | 2 |
| 5 | SKU-4005 | 3 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
lines = order_items.join(orders.select("order_id", "customer_id", "order_ts"), "order_id").where("customer_id <= 2")
so_far = Window.partitionBy("customer_id").orderBy("order_ts", "sku").rowsBetween(Window.unboundedPreceding, 0)
result = lines.select("order_id", "sku", F.size(F.collect_set("sku").over(so_far)).alias("distinct_skus_so_far")).orderBy("order_id", "sku")
result.show()

## Exercise 15: Orders above their customer's average

**Difficulty:** Easy · about 5 min · **Practises:** [Aggregates over a whole partition](pyspark-reference-08-window-functions.ipynb#aggregates-over-a-whole-partition)

*Comparing each row with its group without a self-join.*

Return `order_id`, `customer_id`, `amount` and `customer_avg` (rounded to 2) for orders larger than their customer's average, ordered by `order_id`.

In [ ]:
orders.select("order_id", "customer_id", "amount").show()

**Expected output**

| order_id | customer_id | amount | customer_avg |
| --- | --- | --- | --- |
| 1 | 1 | 120.00 | 78.17 |
| 2 | 1 | 84.50 | 78.17 |
| 4 | 2 | 260.00 | 157.63 |
| 7 | 3 | 310.00 | 229.50 |
| 9 | 4 | 1180.00 | 634.95 |
| 11 | 6 | 210.00 | 153.20 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
with_avg = orders.withColumn("customer_avg", F.round(F.avg("amount").over(Window.partitionBy("customer_id")), 2))
result = with_avg.where(F.col("amount") > F.col("customer_avg")).select("order_id", "customer_id", "amount", "customer_avg").orderBy("order_id")
result.show()

## Exercise 16: Top order per customer in SQL

**Difficulty:** Medium · about 10 min · **Practises:** [Window functions in Spark SQL](pyspark-reference-08-window-functions.ipynb#window-functions-in-spark-sql)

*Spark 4.0 SQL has no QUALIFY; filter window results in an outer query.*

With Spark SQL and a named `WINDOW` clause, return `customer_id`, `order_id`, `amount` of each customer's largest order (ties by `order_id`) and `pct_of_customer` (amount / customer total × 100, rounded to 1), ordered by `customer_id`.

In [ ]:
print("table: orders")

**Expected output**

| customer_id | order_id | amount | pct_of_customer |
| --- | --- | --- | --- |
| 1 | 1 | 120.00 | 51.2 |
| 2 | 4 | 260.00 | 82.5 |
| 3 | 7 | 310.00 | 67.5 |
| 4 | 9 | 1180.00 | 92.9 |
| 5 | 10 | 74.00 | 100.0 |
| 6 | 11 | 210.00 | 68.5 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = spark.sql("""
    WITH ranked AS (
        SELECT customer_id, order_id, amount,
               row_number() OVER w AS rn,
               round(amount / sum(amount) OVER (PARTITION BY customer_id) * 100, 1) AS pct_of_customer
        FROM {o}
        WINDOW w AS (PARTITION BY customer_id ORDER BY amount DESC, order_id)
    )
    SELECT customer_id, order_id, amount, pct_of_customer FROM ranked WHERE rn = 1 ORDER BY customer_id
""", o=orders)
result.show()

**Next:** look anything up in [PySpark reference 8: window functions](pyspark-reference-08-window-functions.ipynb), or test yourself across topics with the [PySpark coding challenges](pyspark-coding-challenges.ipynb).